<a href="https://colab.research.google.com/github/abyanrizz/practicallinearalgebra/blob/main/Chapter_12_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#chapter 12
In this chapter, you will see a few applications of least squares model fitting in real
data. Along the way, you will learn how to implement least squares using several
different—and more numerically stable—Python functions, and you will learn some
new concepts in statistics and machine learning such as multicollinearity, polynomial
regression, and the grid search algorithm as an alternative to least squares.
By the end of this chapter, you will have a deeper understanding of how least squares
is used in applications, including the importance of numerically stable algorithms for
“difficult” situations involving reduced-rank design matrices. And you will see that
the analytic solution provided by least squares outperforms an empirical parameter
search method.
Predicting Bike Rentals Based on Weather
I’m a big fan of bicycles and a big fan of bibimbap (a Korean dish made with rice
and veggies or meat). Therefore, I was happy to find a publicly available dataset about
bike rentals in Seoul.1

The dataset contains nearly nine thousand observations of
data about the number of bikes that were rented in the city and variables about the
weather including temperature, humidity, rainfall, windspeed, and so on.
The purpose of the dataset is to predict the demand for bike sharing based on
weather and season. That is important because it will help bike rental companies and
local governments optimize the availability of healthier modes of transportation. It’s
a great dataset, there’s a lot that could be done with it, and I encourage you to spend

193

time exploring it. In this chapter, I will focus on building relatively simple regression
models to predict bike rental counts based on a few features.
Although this is a book on linear algebra and not statistics, it is still important to
inspect the data carefully before applying and interpreting statistical analyses. The
online code has more details about importing and inspecting the data using the
pandas library. Figure 12-1 shows the data from bike count rentals (the dependent
variable) and rainfall (one of the independent variables).

Figure 12-1. Scatterplots of some data

194 | Chapter 12: Least Squares Applications

Notice that rainfall is a sparse variable—it’s mostly zeros with a relatively small
number of nonzero values. We’ll come back to this in the exercises.
Figure 12-2 shows a correlation matrix from four selected variables. Inspecting corre‐
lation matrices is always a good idea before starting statistical analyses, because it
will show which variables (if any) are correlated and can reveal errors in the data
(e.g., if two supposedly different variables are perfectly correlated). In this case, we
see that bike rental count is positively correlated with hour and temperature (people
rent more bikes later in the day and when the weather is warmer) and negatively
correlated with rainfall. (Note that I’m not showing statistical significance here, so
these interpretations are qualitative.)

Figure 12-2. Correlation matrix of four selected variables
In the first analysis, I want to predict bike rental counts based on rainfall and the
seasons. The seasons (winter, spring, summer, fall) are text labels in the dataset, and
we need to convert them to numbers for the analysis. We could translate the four
seasons into the numbers 1–4, but seasons are circular while regressions are linear.
There are a few ways to deal with this, including using an ANOVA instead of a
regression, using one-hot-encoding (used in deep learning models), or binarizing the
seasons. I’m going to take the latter approach and label autumn and winter “0” and
spring and summer “1”. The interpretation is that a positive beta coefficient indicates
more bike rentals in spring/summer compared to autumn/winter.
(Tangential note: on the one hand, I could have made things simpler by selecting only
continuous variables. But I want to stress that there is more to data science than just
applying a formula to a dataset; there are many nontrivial decisions that affect the
kinds of analyses you can do, and therefore the kinds of results you can obtain.)

Predicting Bike Rentals Based on Weather | 195

The left side of Figure 12-3 shows the design matrix visualized as an image. This is
a common representation of the design matrix, so make sure you are comfortable
interpreting it. The columns are regressors and the rows are observations. Columns
are sometimes normalized to facilitate visual interpretation if the regressors are in
very different numerical scales, although I didn’t do that here. You can see that
rainfall is sparse and that the dataset spans two autumn/winter periods (black areas
in the middle column) and one spring/summer period (white area in the middle).
The intercept is, of course, solid white, because it takes the same value for every
observation.

Figure 12-3. Design matrix and some data
The right side of Figure 12-3 shows the data, plotted as rainfall by bikes rented
separately for the two seasons. Clearly, the data do not lie on a line, because there
are many values at or close to zero on both axes. In other words, visually inspecting
the data suggests that the relationships amongst the variables are nonlinear, which
means that a linear modeling approach might be suboptimal. Again, this highlights
the importance of visually inspecting data and carefully selecting an appropriate
statistical model.
Nonetheless, we will forge ahead using a linear model fit to the data using least
squares. The following code shows how I created the design matrix (variable data is a
pandas dataframe):
196 | Chapter 12: Least Squares Applications

# Create a design matrix and add an intercept
desmat = data[['Rainfall(mm)','Seasons']].to_numpy()
desmat = np.append(desmat,np.ones((desmat.shape[0],1)),axis=1)
# extract DV
y = data[['Rented Bike Count']].to_numpy()
# fit the model to the data using least squares
beta = np.linalg.lstsq(desmat,y,rcond=None)
The beta values for rainfall and season are, respectively, −80 and 369. These numbers
indicate that there are fewer bike rentals when it rains and that there are more bike
rentals in the spring/summer compared to autumn/winter.
Figure 12-4 shows the predicted versus the observed data, separated for the two
seasons. If the model were a perfect fit to the data, the dots would lie on a diagonal
line with a slope of 1. Clearly, that’s not the case, meaning that the model did not
fit the data very well. Indeed, the R
2
is a paltry 0.097 (in other words, the statistical
model accounts for around 1% of the variance in the data). Futhermore, you can see
that the model predicts negative bike rentals, which is not interpretable—bike rental
counts are strictly nonnegative numbers.

Figure 12-4. Scatterplot of predicted by observed data
Thus far in the code, we have received no warnings or errors; we have done nothing
wrong in terms of math or coding. However, the statistical model we used is not the
most appropriate for this research question. You will have the opportunity to improve
it in Exercise 12-1 and Exercise 12-2.

Predicting Bike Rentals Based on Weather | 197

Regression Table Using statsmodels
Without getting too deep into the statistics, I want to show you how to create
a regression table using the statsmodels library. This library works with pandas
dataframes instead of NumPy arrays. The following code shows how to set up and
compute the regression model (OLS stands for ordinary least squares):

In [ ]:
import statsmodels.api as sm
# extract data (staying with pandas dataframes)
desmat_df = data[['Rainfall(mm)','Seasons']]
obsdata_df = data['Rented Bike Count']
# create and fit the model (must explicitly add intercept)
desmat_df = sm.add_constant(desmat_df)
model = sm.OLS(obsdata_df,desmat_df).fit()
print( model.summary()

Multicollinearity
If you’ve taken a statistics course, you might have heard of the term multicollinearity.
The Wikipedia definition is “one predictor variable in a multiple regression model
can be linearly predicted from the others with a substantial degree of accuracy.”2
This means that there are linear dependencies in the design matrix. In the parlance of
linear algebra, multicollinearity is just a fancy term for linear dependence, which is the
same thing as saying that the design matrix is reduced-rank or that it is singular.
A reduced-rank design matrix does not have a left-inverse, which means that the
least squares problem cannot be solved analytically. You will see the implications of
multicollinearity in Exercise 12-3.

Solving GLMs with Multicollinearity

It is, in fact, possible to derive solutions for GLMs that have a reduced-rank design
matrix. This is done using a modification to the QR procedure in the previous
chapter and using the MP pseudoinverse. In the case of reduced-rank design matrix,
there is no unique solution, but we can select the solution with the minimum error.
This is called the minimum norm solution, or simply min-norm, and is used often, for
example in biomedical imaging. Special applications notwithstanding, a design matrix
with linear dependencies usually indicates a problem with the statistical model and
should be investigated (data entry mistakes and coding bugs are common sources of
multicollinearity).
Regularization
Regularization is an umbrella term that refers to various ways of modifying a statis‐
tical model, with the goal of improving numerical stability, transforming singular
or ill-conditioned matrices to full-rank (and thus invertible), or improving generaliz‐
ability by reducing overfitting. There are several forms of regularization depending
on the nature of the problem and the goal of regularizing; some specific techniques
you might have heard of include Ridge (a.k.a. L2), Lasso (a.k.a. L1), Tikhonov, and
shrinkage.
Different regularization techniques work in different ways, but many regularizers
“shift” the design matrix by some amount. You will recall from Chapter 5 that shifting
a matrix means adding some constant to the diagonal as A + λI, and from Chapter 6
that shifting a matrix can transform a reduced-rank matrix into a full-rank matrix.

Predicting Bike Rentals Based on Weather | 199

In this chapter, we will regularize the design matrix by shifting it according to some
proportion of its Frobenius norm. This modifies the least squares solution Equation
12-1.
Equation 12-1. Regularization
β = X
TX + γ ∥ X ∥ F
2
I
−1X
T
y

The key parameter is γ (Greek letter gamma), which determines the amount of
regularization (observe that γ = 0 corresponds to no regularization). Choosing an
appropriate γ parameter is nontrivial, and is often done through statistical techniques
like cross-validation.
The most obvious effect of regularization is that if the design matrix is reduced-rank,
then the regularized squared design matrix is full-rank. Regularization also decreases
the condition number, which measures the “spread” of information in the matrix
(it’s the ratio of the largest to the smallest singular value; you’ll learn about this in
Chapter 14). This increases the numerical stability of the matrix. The statistical impli‐
cation of regularization is to “smooth out” the solution by reducing the sensitivity of
the model to individual data points that might be outliers or nonrepresentative, and
therefore less likely to be observed in new datasets.
Why do I scale by the squared Frobenius norm? Consider that a specified value
of γ, say, γ = .01, can have a huge or a negligible impact on the design matrix
depending on the range of numerical values in the matrix. Therefore, we scale to
the numerical range of the matrix, which means we interpret the γ parameter as
the proportion of regularization. The reason for squaring the Frobenius norm is that
∥ X ∥ F
2
= ∥ XTX ∥ F. In other words, the squared norm of the design matrix equals
the norm of the design matrix times its transpose.
It’s actually more common to use the average of the eigenvalues of the design matrix
instead of the Frobenius norm. After learning about eigenvalues in Chapter 13, you’ll
be able to compare the two regularization methods.
Implementing regularization in code is the focus of Exercise 12-4.
Polynomial Regression
A polynomial regression is like a normal regression but the independent variables are
the x-axis values raised to higher powers. That is, each column i of the design matrix
is defined as x
i
, where x is typically time or space but can be other variables such as
medication dosage or population. The mathematical model looks like this:
y = β0x
0 + β1x
1 + . . . + βnx
n

200 | Chapter 12: Least Squares Applications

Note that x

0 = 1, giving us the intercept of the model. Otherwise, it’s still a regular
regression—the goal is to find the β values that minimize the squared differences
between the predicted and observed data.
The order of the polynomial is the largest power i. For example, a fourth order
polynomial regression has terms up to x
4
(if there is no x
3
term, then it’s still a fourth

order model with β3 = 0).
Figure 12-5 shows an example of the individual regressors and the design matrix
of a third order polynomial (keep in mind that an nth order polynomial has n + 1
regressors including the intercept). The polynomial functions are the basis vectors for
modeling the observed data.